# Final Project

Batch   : CODA-RMT-023 <br>
Topic : Deforestation <br>
Member    : <br>
1. Ryan Farid Pratama as Data Engineer <br>
2. M Hanif Abid as Database & Documentation <br>
3. Narda Noviantoro as Data Analyst <br>

Final project aims to:
1. Data processing to handle data pipeline task ELTL (Extract, Load, Transform and Load) using PySpark
2. Perform data validation with Great Expectations
3. Set up DAGs to automate and schedule the execution of data pipeline task using Apache Airflow
4. Data preparing for SQL Database with PostgreSQL

## A. Pra-Automasi
In this stage, simple data exploration/Exploration Data Analysis (EDA) are performed to understand dataset's condition and determine cleaning steps required before automation. Key notes in this stage:
1. Data Extraction: Spark is used to extract dataset
2. Data Exploration: EDA is performed using Pandas to define data cleaning and processing logic
3. Data Transformation: Drop unused column, drop any duplicate rows if there any
4. Data Validation with Great Expectations
5. Data Loading: Push or loading cleaned DF into GitHub

### A.1 Libraries Import
Libraries import material in order to running a scripts

In [1]:
# Data Manipulation & DataFrame
import pandas as pd
import numpy as np

# Ignore warning to get clean output
import warnings
warnings.filterwarnings("ignore")

# Install the library Great Expectation Package
!pip install -q "great-expectations==0.18.19"           # -q parameter to reduce the output
!pip install great_expectations

from dotenv import load_dotenv
load_dotenv()

# Libraries import & environment setup for pySpark
import os, sys

from pyspark.sql import SparkSession                    # PySpark module import for create session
from pyspark.sql import functions as F                  # PySpark module import for functions/columns manipulation
from pyspark.sql.window import Window                   # PySpark module import for windowing calculations
from pyspark.sql.functions import col, count, when      # PySpark module import for counting total NULL of each column

# Create SparkSession
spark = SparkSession\
    .builder\
    .appName("Python Spark PostgreSQL")\
    .config("spark.jars", "path to jbdc driver .jar file")\
    .getOrCreate()

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


### A.2 Data Extraction
In this sub-stage, dataset is imported using Spark into Spark DataFrame <br>

URL: https://trase.earth/open-data/search?country=INDONESIA <br>
Dataset Name: <br>
1. Main Dataset: Indonesia palm oil - Supply chain data, https://trase.earth/open-data/datasets/supply-chains-indonesia-palm-oil <br>
2. Supporting Dataset: Annual palm deforestation (industrial)(Province), https://trase.earth/open-data/datasets/spatial-metrics-indonesia-palm-oil-palm-oil-deforestation-annual <br>

Chandra, A., Benedict, J., Kamaluddin, A., Putri, S. M., Suavet, C., Gollnow, F., Biddle, H., Mueller, C., Ribeiro, V., & Heilmayr, R. (2026). Indonesia palm oil supply chain: Data and Methods (2018–2024) (Version 1.2.5) [Dataset]. Trase. https://doi.org/10.48650/0717-AJ12

In [2]:
# Import and read the .csv file dataset
# 1. Data Loading Raw Palm Oil into Spark DataFrame Kalimantan only
df_palm_oil = spark.read.csv(r"C:\01. CODA_23_TOR\palm-oil-deforestation-indonesia\dags\palm_oil_data_raw.csv", header=True, inferSchema=True).filter(col("province_of_production").contains("KALIMANTAN"))
df_palm_oil.show(100)

+----+------------------+--------------------+---------------------+----------------------+-----------------------+----------------+--------------------+-------------------+--------------------+--------------+--------------------+-------------------+-------------------------------------+--------------+--------------------+--------------------+-----------------------+--------------------+----------------------+-------------------------------+--------------------------------+------------------------------+--------------------------------+---------------------------------------------+--------------------------------------+-----------------------+---------------------------+-------------------------+----------------------------+-------------------------------+-------------------+------------------+--------------------+
|year|     mill_trase_id|   refinery_trase_id|country_of_production|province_of_production|kabupaten_of_production|    product_type|                mill|         mill_group|

In [3]:
# 2. Data Loading Raw Deforestation into Spark DataFrame Kalimantan only
df_deforestation = spark.read.csv(r"C:\01. CODA_23_TOR\palm-oil-deforestation-indonesia\dags\palm_oil_deforestation_annual_province_raw.csv", header=True, inferSchema=True).filter(col("parent_region")=="KALIMANTAN")
df_deforestation.show(100)

+----+---------+------------+------------------+---------------+-------------+----------------------+-----------------------------------------------+
|year|  country|country_iso2|            region|region_trase_id|parent_region|parent_region_trase_id|annual_palm_deforestation_(industrial)_hectares|
+----+---------+------------+------------------+---------------+-------------+----------------------+-----------------------------------------------+
|2001|INDONESIA|          ID|  KALIMANTAN BARAT|          ID-61|   KALIMANTAN|                 ID-KA|                                         304.45|
|2001|INDONESIA|          ID|KALIMANTAN SELATAN|          ID-63|   KALIMANTAN|                 ID-KA|                                            0.0|
|2001|INDONESIA|          ID| KALIMANTAN TENGAH|          ID-62|   KALIMANTAN|                 ID-KA|                                        2130.03|
|2001|INDONESIA|          ID|  KALIMANTAN TIMUR|          ID-64|   KALIMANTAN|                 ID-KA

### A.3 Data Exploration/EDA (Exploration Data Analysis)
In this sub-stage, sample raw dataset is inspected with EDA process to define data cleaning and processing logic

In [4]:
# 1. Data Viewing Top 5
print(f"--Palm Oil Top 5--")
df_palm_oil.show(5)

print(f"--Deforestation Top 5--")
df_deforestation.show(5)

--Palm Oil Top 5--
+----+------------------+--------------------+---------------------+----------------------+-----------------------+--------------+--------------------+-------------------+-----------+--------------+--------------------+-------------------+-------------------------------------+--------------+--------------------+-------------------+-----------------------+--------------------+----------------------+-------------------------------+--------------------------------+------------------------------+--------------------------------+---------------------------------------------+--------------------------------------+-----------------------+---------------------------+-------------------------+----------------------------+-------------------------------+------------------+------------------+------------------+
|year|     mill_trase_id|   refinery_trase_id|country_of_production|province_of_production|kabupaten_of_production|  product_type|                mill|         mill_grou

In [5]:
# 2. Data Viewing Bottom 5
print(f"--Palm Oil Bottom 5--")
df_palm_oil.tail(5)

print(f"--Deforestation Bottom 5--")
df_deforestation.tail(5)

--Palm Oil Bottom 5--
--Deforestation Bottom 5--


[Row(year=2025, country='INDONESIA', country_iso2='ID', region='KALIMANTAN BARAT', region_trase_id='ID-61', parent_region='KALIMANTAN', parent_region_trase_id='ID-KA', annual_palm_deforestation_(industrial)_hectares=7925.09),
 Row(year=2025, country='INDONESIA', country_iso2='ID', region='KALIMANTAN SELATAN', region_trase_id='ID-63', parent_region='KALIMANTAN', parent_region_trase_id='ID-KA', annual_palm_deforestation_(industrial)_hectares=1042.99),
 Row(year=2025, country='INDONESIA', country_iso2='ID', region='KALIMANTAN TENGAH', region_trase_id='ID-62', parent_region='KALIMANTAN', parent_region_trase_id='ID-KA', annual_palm_deforestation_(industrial)_hectares=3293.11),
 Row(year=2025, country='INDONESIA', country_iso2='ID', region='KALIMANTAN TIMUR', region_trase_id='ID-64', parent_region='KALIMANTAN', parent_region_trase_id='ID-KA', annual_palm_deforestation_(industrial)_hectares=4023.29),
 Row(year=2025, country='INDONESIA', country_iso2='ID', region='KALIMANTAN UTARA', region_tra

In [6]:
# 3. Verify Row Counts
print(f"Total Palm Oil Raw Rows: {df_palm_oil.count()}")
print(f"Total Deforestation Raw Rows : {df_deforestation.count()}")

Total Palm Oil Raw Rows: 588476
Total Deforestation Raw Rows : 125


In [7]:
# 4. Ensure all of data type in PySpark align with DDL design before load it into Database
# 4.1. Palm Oil Table
print(f"--Palm Oil Data Type--")
df_palm_oil.printSchema()

# 4.2. Deforestation Table
print(f"--Deforestation Data Type--")
df_deforestation.printSchema()

--Palm Oil Data Type--
root
 |-- year: integer (nullable = true)
 |-- mill_trase_id: string (nullable = true)
 |-- refinery_trase_id: string (nullable = true)
 |-- country_of_production: string (nullable = true)
 |-- province_of_production: string (nullable = true)
 |-- kabupaten_of_production: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- mill: string (nullable = true)
 |-- mill_group: string (nullable = true)
 |-- refinery: string (nullable = true)
 |-- refinery_group: string (nullable = true)
 |-- exporter: string (nullable = true)
 |-- exporter_group: string (nullable = true)
 |-- zero_deforestation_indonesia_palm_oil: string (nullable = true)
 |-- port_of_export: string (nullable = true)
 |-- importer: string (nullable = true)
 |-- importer_group: string (nullable = true)
 |-- country_of_first_import: string (nullable = true)
 |-- economic_bloc_name: string (nullable = true)
 |-- economic_bloc_trase_id: string (nullable = true)
 |-- province_of_producti

In [8]:
# 5. Count NULL data
# 5.1 Palm Oil Table
print(f"--Palm Oil Total Null Data--")
df_palm_oil.select([count(when(col(c).isNull(), c)).alias(c) 
                     for c in df_palm_oil.columns]).show()

# 5.2 Deforestation Table
print(f"--Deforestation Total Null Data--")
df_deforestation.select([count(when(col(c).isNull(), c)).alias(c) 
                     for c in df_deforestation.columns]).show()

--Palm Oil Total Null Data--
+----+-------------+-----------------+---------------------+----------------------+-----------------------+------------+----+----------+--------+--------------+--------+--------------+-------------------------------------+--------------+--------+--------------+-----------------------+------------------+----------------------+-------------------------------+--------------------------------+------------------------------+--------------------------------+---------------------------------------------+--------------------------------------+-----------------------+---------------------------+-------------------------+----------------------------+-------------------------------+------+---+-----------+
|year|mill_trase_id|refinery_trase_id|country_of_production|province_of_production|kabupaten_of_production|product_type|mill|mill_group|refinery|refinery_group|exporter|exporter_group|zero_deforestation_indonesia_palm_oil|port_of_export|importer|importer_group|countr

There is no null data in both dataset, so there is no need to fill

In [9]:
# 6. Count distinct rows
# 6.1 Palm Oil table distinct rows
print(f"--Palm Oil Total Distinct Rows: {df_palm_oil.distinct().count()}")
if df_palm_oil.count() == df_palm_oil.distinct().count():
    print("All rows is unique")

# 6.2 Deforestation table distinct rows
print(f"--Deforestation Total Distinct Rows: {df_deforestation.distinct().count()}")
if df_deforestation.count() == df_deforestation.distinct().count():
    print("All rows is unique")

--Palm Oil Total Distinct Rows: 588476
All rows is unique
--Deforestation Total Distinct Rows: 125
All rows is unique


In [ ]:
# 1. Filter only contain "KALIMANTAN" in province
df_kalimantan = df_palm_oil.filter(
    F.col("province_of_production").rlike("KALIMANTAN")
    )

# 2. Count sum the most common value of each province
df_grouped = (
    df_kalimantan
    .groupBy("province_of_production")
    .agg(F.count("*").alias("jumlah_baris"))
    .orderBy(F.col("jumlah_baris").desc())
)

# 3. Showing the results
df_grouped.show(truncate=False)

+----------------------+------------+
|province_of_production|jumlah_baris|
+----------------------+------------+
|KALIMANTAN TENGAH     |205582      |
|KALIMANTAN BARAT      |161961      |
|KALIMANTAN TIMUR      |142390      |
|KALIMANTAN SELATAN    |56296       |
|KALIMANTAN UTARA      |22247       |
+----------------------+------------+



### A.4 Conclusion Criteria
In this sub-stage, data cleaning conclusions for transform.py is formulated. Conclusion criteria after EDA is performed are:
1. Filter out/drop rows unrelevant ones
2. Ensure and change numerical column data type like VOLUME, FOB and OIL_PALM_HA into FloatType or DoubleType in PySpark
3. Drop unused column based on data dictionaries

### A.5 Data Cleaning
In this sub-stage, data cleaning of raw table is performed using pySpark. Data cleaning in this process including:
1. Whitelist approach, define which columns to keep based on data dictionaries from main dataset (palm oil)
2. Blacklist approach, define which unused columns to drop

In [11]:
# 1.1 Select to keep from main dataset
keep_columns = ["year", 
                "province_of_production", "province_of_production_trase_id", "kabupaten_of_production", 
                "product_type", "exporter_group", 
                "zero_deforestation_indonesia_palm_oil", "country_of_first_import", 
                "palm_oil_deforestation_annual_exposure", "volume", "total_emission_exposure"]
df_palm_oil_cleaned = df_palm_oil[keep_columns]
df_palm_oil_cleaned.show(10)

print(f"Total Palm Oil Raw Rows: {df_palm_oil_cleaned.count()}") # Verify Row Counts

+----+----------------------+-------------------------------+-----------------------+--------------+-------------------+-------------------------------------+-----------------------+--------------------------------------+------------------+-----------------------+
|year|province_of_production|province_of_production_trase_id|kabupaten_of_production|  product_type|     exporter_group|zero_deforestation_indonesia_palm_oil|country_of_first_import|palm_oil_deforestation_annual_exposure|            volume|total_emission_exposure|
+----+----------------------+-------------------------------+-----------------------+--------------+-------------------+-------------------------------------+-----------------------+--------------------------------------+------------------+-----------------------+
|2018|      KALIMANTAN BARAT|                          ID-61|               KETAPANG|CRUDE PALM OIL|         WINGS AGRO|                                 NONE|                  INDIA|                  0.057

In [12]:
# 1.2 Ensure data type
df_palm_oil_cleaned.printSchema()

root
 |-- year: integer (nullable = true)
 |-- province_of_production: string (nullable = true)
 |-- province_of_production_trase_id: string (nullable = true)
 |-- kabupaten_of_production: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- exporter_group: string (nullable = true)
 |-- zero_deforestation_indonesia_palm_oil: string (nullable = true)
 |-- country_of_first_import: string (nullable = true)
 |-- palm_oil_deforestation_annual_exposure: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- total_emission_exposure: double (nullable = true)



In [13]:
# 1.3 Drop/clean with zero value


In [14]:
# 2. Drop columns from supporting dataset
drop_columns = ["country", "country_iso2", "parent_region", "parent_region_trase_id"]

df_deforestation_cleaned = df_deforestation.drop(*drop_columns)
df_deforestation_cleaned.show(10)

print(f"Total Palm Oil Raw Rows: {df_deforestation_cleaned.count()}") # Verify Row Counts

+----+------------------+---------------+-----------------------------------------------+
|year|            region|region_trase_id|annual_palm_deforestation_(industrial)_hectares|
+----+------------------+---------------+-----------------------------------------------+
|2001|  KALIMANTAN BARAT|          ID-61|                                         304.45|
|2001|KALIMANTAN SELATAN|          ID-63|                                            0.0|
|2001| KALIMANTAN TENGAH|          ID-62|                                        2130.03|
|2001|  KALIMANTAN TIMUR|          ID-64|                                          19.48|
|2001|  KALIMANTAN UTARA|          ID-65|                                        1020.24|
|2002|  KALIMANTAN BARAT|          ID-61|                                          212.5|
|2002|KALIMANTAN SELATAN|          ID-63|                                          11.33|
|2002| KALIMANTAN TENGAH|          ID-62|                                        3989.45|
|2002|  KA

In [15]:
# Ensure data type
df_deforestation_cleaned.printSchema()

root
 |-- year: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- region_trase_id: string (nullable = true)
 |-- annual_palm_deforestation_(industrial)_hectares: double (nullable = true)



### A.6 Data Loading
In this sub-stage, data cleaned is:
1. Convert into Pandas DataFrame
2. Export dataset into .csv
3. Load into repositories

In [16]:
# 1. Convert main dataset into Pandas DataFrame
df_palm_oil_pandas = df_palm_oil_cleaned.toPandas()

# 2. Convert supporting dataset into Pandas DataFrame
df_deforestation_pandas = df_deforestation_cleaned.toPandas()

# 3. Load main dataset into .csv
df_palm_oil_pandas.to_csv("palm_oil_cleaned.csv", index=False)

# 4. Load supporting dataset into .csv
df_deforestation_pandas.to_csv("deforestation_cleaned.csv", index=False)

In [17]:
df_palm_oil_pandas.describe()

,year,palm_oil_deforestation_annual_exposure,volume,total_emission_exposure
count,588476.000000,588476.000000,5.884760e+05,5.884760e+05
mean,2021.039657,0.292992,2.220536e+02,1.042200e+03
std,1.777634,20.682422,7.382199e+03,6.261230e+04
min,2018.000000,0.000000,1.000279e-03,0.000000e+00
25%,2020.000000,0.000000,1.790798e+00,0.000000e+00
50%,2021.000000,0.000000,9.662354e+00,4.902925e-01
75%,2023.000000,0.000757,4.474494e+01,4.336164e+01
max,2024.000000,7674.273185,1.440255e+06,2.309235e+07


In [18]:
df_deforestation_pandas.describe()

,year,annual_palm_deforestation_(industrial)_hectares
count,125.000000,125.000000
mean,2013.000000,11757.243520
std,7.240121,13787.892154
min,2001.000000,0.000000
25%,2007.000000,2618.950000
50%,2013.000000,5814.430000
75%,2019.000000,16924.110000
max,2025.000000,67071.630000


In [19]:
df_deforestation_pandas["region"].value_counts()

region
KALIMANTAN BARAT      25
KALIMANTAN SELATAN    25
KALIMANTAN TENGAH     25
KALIMANTAN TIMUR      25
KALIMANTAN UTARA      25
Name: count, dtype: int64

### A.7 Data Validation with Great Expectations

#### A.7.1 Instantiate Data Context
A Data Context is the primary entry point for a Great Expectations (GX) deployment, and it provides the configurations and methods for all supporting GX components. As the primary entry point for the GX API, the Data Context provides a convenient method for accessing common objects based on untyped input or common defaults. A Data Context also allows us to configure top-level components, and we can use different storage methodologies to back up our Data Context configuration.

In [20]:
# Create a data context using the updated method
import great_expectations as gx
import great_expectations.expectations as gxe
context = gx.get_context(mode='file', project_root_dir='./')

#### A.7.2 Connect to A Datasource

Datasource : provides a standard API for accessing and interacting with data from a wide variety of source systems.
Data Asset : a collection of records within a Datasource which is usually named based on the underlying data system and sliced to correspond to a desired specification.

In [21]:
# 1. Load data cleaned result
raw_csv_path_1 = "palm_oil_cleaned.csv"                    # Path for palm oil dataset
df_cleaned_1 = pd.read_csv(raw_csv_path_1)          

raw_csv_path_2 = "deforestation_cleaned.csv"               # Path for deforestation dataset
df_cleaned_2 = pd.read_csv(raw_csv_path_2)          

# 2. Give a name to a Datasource. This name must be unique between Datasources
# 2.1 Datasource 1 Palm Oil
datasource_name_1 = "csv-raw-data-palm-oil"
asset_name_1 = "palm-oil-asset"
path_to_data_1 = raw_csv_path_1
batch_name_1 = "batch-definition-palm-oil"

# 2.1 Datasource 2 Deforestation
datasource_name_2 = "csv-raw-data-deforestation"
asset_name_2 = "deforestation-asset"
path_to_data_2 = raw_csv_path_2
batch_name_2 = "batch-definition-deforestation"

# 3. Give a name to a data asset
# 3.1 Palm Oil
datasource_1 = context.data_sources.add_pandas(datasource_name_1)

# 3.2 Deforestation
datasource_2 = context.data_sources.add_pandas(datasource_name_2)

# 4. Build batch definition
# 4.1 Palm Oil
asset_1 = datasource_1.add_dataframe_asset(asset_name_1)
batch_definition_1 = asset_1.add_batch_definition_whole_dataframe(batch_name_1)

# 4.2 Deforestation
asset_2 = datasource_2.add_dataframe_asset(asset_name_2)
batch_definition_2 = asset_2.add_batch_definition_whole_dataframe(batch_name_2)

# 5. Read the csv file (same path as before) and prepare the batch parameters
# 5.1 Palm Oil
df_cleaned_1 = pd.read_csv(path_to_data_1)
batch_parameters_1 = {"dataframe": df_cleaned_1}

# 5.2 Deforestation
df_cleaned_2 = pd.read_csv(path_to_data_2)
batch_parameters_2 = {"dataframe": df_cleaned_2}

#### A.7.3 Expectation for Main Dataset

##### A.7.3.1 Create an Expectation Suite
An Expectation Suite is a collection of verifiable assertions about data. Expectation Suites combine multiple Expectations into an overall description of data.
Expectation Suite names are customizable, and the only constraint is that it must be unique to a given project.
We'll use a Validator to interact with your batch of data and generate an Expectation Suite.

In [22]:
# Expectation Suite for Palm Oil
expectation_suite_name_1 = "expectation-palm-oil-dataset"
suite_1 = context.suites.add(gx.ExpectationSuite(name=expectation_suite_name_1))

# Get a batch of data using the batch definition above
batch_1 = batch_definition_1.get_batch(batch_parameters=batch_parameters_1)

# We will collect all expectations in this list, then save them into the suite
expectations_1 = []

##### A.7.3.2 Expectations
An Expectation is a verifiable assertion about source data. Similar to assertions in traditional Python unit tests, Expectations provide a flexible, declarative language for describing expected behaviors.Expectations is basically what do we expect from the data.
For example we expect our columns:
1. **to be in set** <br>
Expect the set of distinct column values to be contained by a given set, which is "KALIMANTAN"
2. **most common value to be in set** <br>
Expect the most common value to be within the designated value set.
3. **to exist** <br>
Checks for the existence of a specified column within a table.
4. **value to be between** <br>
Expect the column entries to be between a minimum value and a maximum value (inclusive), which is year
5. **value lengths to be between** <br>
Expect the column entries to be strings with length between a minimum value and a maximum value (inclusive).

In [ ]:
# 1. Expectation 1 : to be in set
# Ensures that the product_type column only contains valid, expected categories ("CRUDE PALM OIL" or "REFINED PALM OIL") without unauthorized entries.
expectation_1 = gxe.ExpectColumnValuesToBeInSet(column="product_type", value_set=["CRUDE PALM OIL", "REFINED PALM OIL"])
expectations_1.append(expectation_1)
batch_1.validate(expectation_1)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 33.03it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "product_type",
      "value_set": [
        "CRUDE PALM OIL",
        "REFINED PALM OIL"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 588476,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# 2. Expectation 2  : most common value to be in set
# Verifies that the dominant or most frequently occurring production province (province_of_production) aligns with the expected primary region ("KALIMANTAN TENGAH").
expectation_2 = gxe.ExpectColumnMostCommonValueToBeInSet(column="province_of_production", value_set=["KALIMANTAN TENGAH"])
expectations_1.append(expectation_2)
batch_1.validate(expectation_2)

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 49.91it/s]  


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_most_common_value_to_be_in_set",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "province_of_production",
      "value_set": [
        "KALIMANTAN TENGAH"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": [
      "KALIMANTAN TENGAH"
    ]
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# 3. Expectation 3  : to exist
# Confirms the physical existence and proper positioning of the province_of_production_trase_id column within the table schema.
expectation_3 = gxe.ExpectColumnToExist(column="province_of_production_trase_id", column_index=2)
expectations_1.append(expectation_3)
batch_1.validate(expectation_3)

Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 379.25it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_to_exist",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "province_of_production_trase_id",
      "column_index": 2
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# 4. Expectation 4  : value to be between
# Validates that all records in the year column fall strictly within the expected temporal study window (2018–2024).
expectation_4 = gxe.ExpectColumnValuesToBeBetween(column="year", min_value=2018, max_value=2024)
expectations_1.append(expectation_4)
batch_1.validate(expectation_4)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 80.36it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "year",
      "min_value": 2018.0,
      "max_value": 2024.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 588476,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# 5. Expectation 5  : value lengths to be between
# Checks that the string length of the identifiers in province_of_production_trase_id stays within a valid character range (1 to 5 characters).
expectation_5 = gxe.ExpectColumnValueLengthsToBeBetween(column="province_of_production_trase_id", min_value=1, max_value=5)
expectations_1.append(expectation_5)
batch_1.validate(expectation_5)

Calculating Metrics: 100%|██████████| 11/11 [00:00<00:00, 22.70it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_value_lengths_to_be_between",
    "kwargs": {
      "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
      "column": "province_of_production_trase_id",
      "min_value": 1,
      "max_value": 5
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 588476,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

##### A.7.3.2 Save Suite & Run Checkpoint
A Checkpoint is the primary means for validating data in a production deployment of Great Expectations.

Checkpoints provide a convenient abstraction for bundling the Validation of a Batch (or Batches) of data against an Expectation Suite (or several), as well as the Actions that should be taken after the validation.

In [28]:
# 1. Save all the expectations into Suite
for exp in expectations_1:
    suite_1.add_expectation(exp)
suite_1.save()

In [29]:
# 2. Create Validation Definition
name_validation = "validation-definition-palm-oil"
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(name=name_validation, data=batch_definition_1,suite=suite_1))

In [30]:
# 3. Create and Execute Checkpoint
checkpoint_1 = context.checkpoints.add(
    gx.Checkpoint(name="palm_oil_checkpoint", validation_definitions=[validation_definition], actions=[gx.checkpoint.actions.UpdateDataDocsAction(name="update_data_docs")]))

In [31]:
checkpoint_result_1 = checkpoint_1.run(batch_parameters=batch_parameters_1)
print(checkpoint_result_1.describe())

Calculating Metrics: 100%|██████████| 26/26 [00:00<00:00, 29.21it/s] 


{
    "success": true,
    "statistics": {
        "evaluated_validations": 1,
        "success_percent": 100.0,
        "successful_validations": 1,
        "unsuccessful_validations": 0
    },
    "validation_results": [
        {
            "success": true,
            "statistics": {
                "evaluated_expectations": 5,
                "successful_expectations": 5,
                "unsuccessful_expectations": 0,
                "success_percent": 100.0
            },
            "expectations": [
                {
                    "expectation_type": "expect_column_values_to_be_in_set",
                    "success": true,
                    "kwargs": {
                        "batch_id": "csv-raw-data-palm-oil-palm-oil-asset",
                        "column": "product_type",
                        "value_set": [
                            "CRUDE PALM OIL",
                            "REFINED PALM OIL"
                        ]
                    },
              

#### A.7.4 Expectation for Supporting Dataset

##### A.7.4.1 Create an Expectation Suite
An Expectation Suite is a collection of verifiable assertions about data. Expectation Suites combine multiple Expectations into an overall description of data.
Expectation Suite names are customizable, and the only constraint is that it must be unique to a given project.
We'll use a Validator to interact with your batch of data and generate an Expectation Suite.

In [54]:
# Expectation Suite for Deforestation
expectation_suite_name_2 = "expectation-deforestation-dataset"
suite_2 = context.suites.add(gx.ExpectationSuite(name=expectation_suite_name_2))

# Get a batch of data using the batch definition above
batch_2 = batch_definition_2.get_batch(batch_parameters=batch_parameters_2)

# We will collect all expectations in this list, then save them into the suite
expectations_2 = []

##### A.7.4.2 Expectations
An Expectation is a verifiable assertion about source data. Similar to assertions in traditional Python unit tests, Expectations provide a flexible, declarative language for describing expected behaviors.Expectations is basically what do we expect from the data.
For example we expect our columns:
1. **to be in set** <br>
Expect the set of distinct column values to be contained by a given set.
2. **distinct values to contain set** <br>
Expect the set of distinct column values to contain a given set.
3. **value lengths to be between** <br>
Expect the column entries to be strings with length between a minimum value and a maximum value (inclusive).

In [ ]:
# 1. Expectation 1 : to be in set
# Ensures that all values in the region column are restricted to the five valid Kalimantan provinces without unauthorized entries.
expectation_A = gxe.ExpectColumnValuesToBeInSet(column="region", value_set=["KALIMANTAN UTARA", "KALIMANTAN SELATAN", "KALIMANTAN TENGAH", "KALIMANTAN BARAT", "KALIMANTAN TIMUR"])
expectations_2.append(expectation_A)
batch_2.validate(expectation_A)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 491.12it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "batch_id": "csv-raw-data-deforestation-deforestation-asset",
      "column": "region",
      "value_set": [
        "KALIMANTAN UTARA",
        "KALIMANTAN SELATAN",
        "KALIMANTAN TENGAH",
        "KALIMANTAN BARAT",
        "KALIMANTAN TIMUR"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 125,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# 2. Expectation 2 : distinct values to contain set
# Verifies that the distinct values in the year column successfully contain the required temporal range (2018–2024), while permitting historical data outside that window.
expectation_B = gxe.ExpectColumnDistinctValuesToContainSet(column="year", value_set=[2018, 2019, 2020, 2021, 2022, 2023, 2024])
expectations_2.append(expectation_B)
batch_2.validate(expectation_B)

Calculating Metrics: 100%|██████████| 5/5 [00:00<00:00, 429.13it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_distinct_values_to_contain_set",
    "kwargs": {
      "batch_id": "csv-raw-data-deforestation-deforestation-asset",
      "column": "year",
      "value_set": [
        2018,
        2019,
        2020,
        2021,
        2022,
        2023,
        2024
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": null,
    "missing_count": 0,
    "partial_missing_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# 3. Expectation 3 : value lengths to be between
# Checks that the string length of the identifiers in region_trase_id falls within the valid character length range (1 to 5 characters).
expectation_C = gxe.ExpectColumnValueLengthsToBeBetween(column="region_trase_id", min_value=1, max_value=5)
expectations_2.append(expectation_C)
batch_2.validate(expectation_C)

Calculating Metrics: 100%|██████████| 11/11 [00:00<00:00, 338.26it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_value_lengths_to_be_between",
    "kwargs": {
      "batch_id": "csv-raw-data-deforestation-deforestation-asset",
      "column": "region_trase_id",
      "min_value": 1,
      "max_value": 5
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 125,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

##### A.7.4.3 Save Suite & Run Checkpoint
A Checkpoint is the primary means for validating data in a production deployment of Great Expectations.

Checkpoints provide a convenient abstraction for bundling the Validation of a Batch (or Batches) of data against an Expectation Suite (or several), as well as the Actions that should be taken after the validation.

In [58]:
# 1. Save all the expectations into Suite
for exp in expectations_2:
    suite_2.add_expectation(exp)
suite_2.save()

In [59]:
# 2. Create Validation Definition
name_validation = "validation-definition-deforestation"
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(name=name_validation, data=batch_definition_2,suite=suite_2))

In [60]:
# 3. Create and Execute Checkpoint
checkpoint_2 = context.checkpoints.add(
    gx.Checkpoint(name="deforestation_checkpoint", validation_definitions=[validation_definition], actions=[gx.checkpoint.actions.UpdateDataDocsAction(name="update_data_docs")]))

In [61]:
checkpoint_result_2 = checkpoint_2.run(batch_parameters=batch_parameters_2)
print(checkpoint_result_2.describe())

Calculating Metrics: 100%|██████████| 20/20 [00:00<00:00, 929.74it/s] 


{
    "success": true,
    "statistics": {
        "evaluated_validations": 1,
        "success_percent": 100.0,
        "successful_validations": 1,
        "unsuccessful_validations": 0
    },
    "validation_results": [
        {
            "success": true,
            "statistics": {
                "evaluated_expectations": 3,
                "successful_expectations": 3,
                "unsuccessful_expectations": 0,
                "success_percent": 100.0
            },
            "expectations": [
                {
                    "expectation_type": "expect_column_values_to_be_in_set",
                    "success": true,
                    "kwargs": {
                        "batch_id": "csv-raw-data-deforestation-deforestation-asset",
                        "column": "region",
                        "value_set": [
                            "KALIMANTAN UTARA",
                            "KALIMANTAN SELATAN",
                            "KALIMANTAN TENGAH",
     

### A.8 Data Loading

In this final stage, we will loading our cleaned PySpark DataFrames into PostgreSQL Data Warehouse with following this steps:
1. Load Cleaned Main Dataset Tables
2. Load Cleaned Supporting Dataset Tables

In [63]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Load credentials dari file .env
load_dotenv()

DB_USER = os.getenv("PG_USER", "postgres")
DB_PASSWORD = os.getenv("PG_PASSWORD", "Architect!79")
DB_HOST = os.getenv("PG_HOST", "localhost")
DB_PORT = os.getenv("PG_PORT", "5432")
DB_NAME = os.getenv("PG_DB", "fp23_palm_oil")  # Sesuaikan dengan nama DB kalian

# 2. Buat SQLAlchemy Engine
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Load Main Dataset (Palm Oil)
print("Loading stg_palm_oil...")
df_palm_oil_pandas.to_sql(
    name="stg_palm_oil",
    con=engine,
    if_exists="replace",  # Gunakan 'replace' saat testing awal, atau 'append' jika schema sudah dibuatkan Role 2
    index=False
)
print("✅ stg_palm_supply_chain BERHASIL di-load!")

# Load Secondary Dataset (Deforestation)
print("Loading stg_palm_deforestation...")
df_deforestation_pandas.to_sql(
    name="stg_palm_deforestation",
    con=engine,
    if_exists="replace",
    index=False
)
print("✅ stg_palm_deforestation BERHASIL di-load!")

print("🚀 SELURUH DATASET BERHASIL DI-LOAD KE POSTGRESQL!")

Loading stg_palm_oil...
✅ stg_palm_supply_chain BERHASIL di-load!
Loading stg_palm_deforestation...
✅ stg_palm_deforestation BERHASIL di-load!
🚀 SELURUH DATASET BERHASIL DI-LOAD KE POSTGRESQL!


In [64]:
spark.stop()